# Canadian Sheep Federation Data Analyst Assessment
## Premium ETL + Climate-Cheese Intelligence Report

### Executive Overview
This notebook delivers a **production-style data pipeline** and a polished analytical narrative that investigates the relationship between provincial climate and cheese production patterns in Canada. The solution integrates three source files, applies rigorous sanitization standards, and builds a unified analytical layer before generating executive-grade visualizations.

**Core question:** *Do provincial weather conditions influence cheese profiles and milk-type production patterns, and if so, how strongly compared with regional production culture?*

In [1]:
from __future__ import annotations

import re
import unicodedata
from pathlib import Path
from typing import Dict, Iterable, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## Methodology

### Data Engineering Design
- **Extract:** Load raw cheese and weather sources with robust path fallback logic.
- **Transform:** Standardize province identifiers (codes + names), parse mixed-format temperature fields, clean numeric features, and engineer business-ready metrics.
- **Load:** Produce a validated analytical table by joining cleaned cheese records to province-level climate aggregates.

### Data Sanitization Standards
- **Regex and text normalization** are used to handle whitespace, punctuation, special characters, and French/English province naming variants.
- **Numerical null handling** follows transparent rules:
  - `MoisturePercent`: province-median imputation, then global median fallback.
  - Temperature features: parsed from string fields and aggregated with missing-safe averaging.
- **Integrity controls** include merge cardinality validation and row-count preservation checks.

In [2]:
# ------------------------
# Global mappings/constants
# ------------------------
PROVINCE_CODE_TO_NAME: Dict[str, str] = {
    "AB": "Alberta",
    "BC": "British Columbia",
    "MB": "Manitoba",
    "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia",
    "NT": "Northwest Territories",
    "NU": "Nunavut",
    "ON": "Ontario",
    "PE": "Prince Edward Island",
    "QC": "Quebec",
    "SK": "Saskatchewan",
    "YT": "Yukon",
}

PROVINCE_NAME_ALIASES: Dict[str, str] = {
    "alberta": "Alberta",
    "british columbia": "British Columbia",
    "colombie britannique": "British Columbia",
    "manitoba": "Manitoba",
    "new brunswick": "New Brunswick",
    "nouveau brunswick": "New Brunswick",
    "newfoundland and labrador": "Newfoundland and Labrador",
    "terre neuve and labrador": "Newfoundland and Labrador",
    "terre neuve et labrador": "Newfoundland and Labrador",
    "nova scotia": "Nova Scotia",
    "nouvelle ecosse": "Nova Scotia",
    "ontario": "Ontario",
    "prince edward island": "Prince Edward Island",
    "ile du prince edouard": "Prince Edward Island",
    "quebec": "Quebec",
    "saskatchewan": "Saskatchewan",
    "northwest territories": "Northwest Territories",
    "territoires du nord ouest": "Northwest Territories",
    "nunavut": "Nunavut",
    "yukon": "Yukon",
}

FAT_PROXY_MAP: Dict[str, float] = {
    "lower fat": 25.0,
    "higher fat": 45.0,
}

PREMIUM_TEMPLATE = "simple_white"
PREMIUM_FONT = "Inter, Segoe UI, Arial"
PREMIUM_BG = "#f8fafc"
GRID_COLOR = "#dbe2ea"


def normalize_text(value: str) -> str:
    """Normalize text for reliable matching (accents, punctuation, spacing)."""
    if pd.isna(value):
        return np.nan
    txt = str(value).strip()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("utf-8")
    txt = txt.lower()
    txt = re.sub(r"[^a-z0-9\s\-&]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


def normalize_province_name(name: str) -> str:
    """Map French/English/variant province names to one canonical English form."""
    key = normalize_text(name)
    if pd.isna(key):
        return np.nan
    return PROVINCE_NAME_ALIASES.get(key, str(name).strip().title())


def parse_celsius_from_text(value: str) -> float:
    """Extract the first Celsius numeric value from mixed-format text fields."""
    if pd.isna(value):
        return np.nan
    cleaned = str(value).replace("−", "-")
    match = re.search(r"-?\d+(?:\.\d+)?", cleaned)
    return float(match.group()) if match else np.nan


def classify_milk_type(value: str) -> str:
    """Collapse raw milk labels into clean analysis groups."""
    if pd.isna(value):
        return "Unknown"
    label = str(value).lower()
    has_cow = "cow" in label
    has_goat = "goat" in label
    has_sheep = ("ewe" in label) or ("sheep" in label)
    has_buffalo = "buffalo" in label

    n_sources = sum([has_cow, has_goat, has_sheep, has_buffalo])
    if n_sources > 1:
        return "Mixed"
    if has_sheep:
        return "Sheep"
    if has_goat:
        return "Goat"
    if has_cow:
        return "Cow"
    if has_buffalo:
        return "Buffalo"
    return "Other"


def resolve_temperature_source(base_dir: Path) -> Path:
    """Return path to temperature dataset (prefers CSV, falls back to ZIP)."""
    csv_path = base_dir / "Canada_Temperature_Data.csv"
    zip_path = base_dir / "Canada_Temperature_Data.csv.zip"
    if csv_path.exists():
        return csv_path
    if zip_path.exists():
        return zip_path
    raise FileNotFoundError("Neither Canada_Temperature_Data.csv nor Canada_Temperature_Data.csv.zip was found.")


def load_data(base_dir: str = ".") -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load cheese data and two weather sources from disk."""
    root = Path(base_dir)
    cheese_df = pd.read_csv(root / "cheese_data.csv")
    weather_df = pd.read_csv(root / "canada_weather.csv")

    temp_source = resolve_temperature_source(root)
    if temp_source.suffix.lower() == ".zip":
        temp_df = pd.read_csv(temp_source, compression="zip")
    else:
        temp_df = pd.read_csv(temp_source)

    return cheese_df, weather_df, temp_df


def clean_cheese_data(cheese_df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean cheese dataset with province harmonization and numeric conditioning.

    Steps:
    1) Normalize province codes/names using regex and alias mapping.
    2) Handle null numerical values in moisture via robust imputation.
    3) Engineer analytical fat and milk-type features for visualization.
    """
    df = cheese_df.copy()

    # Province code normalization from source field.
    df["ProvinceCode"] = (
        df["ManufacturerProvCode"]
        .astype(str)
        .str.upper()
        .str.replace(r"[^A-Z]", "", regex=True)
        .str.strip()
    )

    # Canonical English province names to align with climate data.
    df["Province"] = df["ProvinceCode"].map(PROVINCE_CODE_TO_NAME)
    if "Province" in cheese_df.columns:
        fallback_names = cheese_df["Province"].apply(normalize_province_name)
        df["Province"] = df["Province"].fillna(fallback_names)

    # Moisture cleaning: province median then global median.
    df["MoisturePercent"] = pd.to_numeric(df["MoisturePercent"], errors="coerce")
    df["MoisturePercent"] = df.groupby("ProvinceCode")["MoisturePercent"].transform(
        lambda s: s.fillna(s.median())
    )
    df["MoisturePercent"] = df["MoisturePercent"].fillna(df["MoisturePercent"].median())

    # FatLevel in source is categorical; map to numeric proxy for bubble analysis.
    df["FatLevelClean"] = df["FatLevel"].astype(str).str.lower().str.strip()
    df["FatPercentEstimate"] = df["FatLevelClean"].map(FAT_PROXY_MAP).fillna(35.0)

    # Milk feature engineering.
    df["MilkTypeGroup"] = df["MilkTypeEn"].apply(classify_milk_type)

    # Drop records that cannot support province-level climate merge.
    df = df.dropna(subset=["CheeseName", "ProvinceCode", "Province"]).copy()
    df = df[df["ProvinceCode"].isin(PROVINCE_CODE_TO_NAME.keys())].reset_index(drop=True)

    return df


def clean_weather_data(weather_df: pd.DataFrame, temp_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a reliable province-level AverageAnnualTemperature by fusing two weather datasets.

    - Source A: community climate summary (`canada_weather.csv`)
    - Source B: historical station monthly means (`Canada_Temperature_Data`)
    """
    # ---- Source A (community summaries) ----
    w = weather_df.copy()
    annual_high_col = next(c for c in w.columns if "Annual(Avg. high" in c)
    annual_low_col = next(c for c in w.columns if "Annual(Avg. low" in c)

    w["AnnualHighC"] = w[annual_high_col].apply(parse_celsius_from_text)
    w["AnnualLowC"] = w[annual_low_col].apply(parse_celsius_from_text)
    w["WeatherAnnualMeanC"] = (w["AnnualHighC"] + w["AnnualLowC"]) / 2

    w["ProvinceCode"] = (
        w["Community"].astype(str).str.extract(r",\s*([A-Z]{2})\s*$")[0].str.upper().str.strip()
    )

    weather_agg = (
        w.dropna(subset=["ProvinceCode", "WeatherAnnualMeanC"])
        .groupby("ProvinceCode", as_index=False)["WeatherAnnualMeanC"]
        .mean()
    )

    # ---- Source B (station monthly means) ----
    t = temp_df.copy()
    t["ProvinceCode"] = t["Prov"].astype(str).str.upper().str.strip()
    t["Tm"] = pd.to_numeric(t["Tm"], errors="coerce")

    # Data quality filters.
    t = t[t["Tm"].between(-50, 40, inclusive="both")]
    if "Year" in t.columns:
        t = t[t["Year"] >= 1991]

    station_agg = (
        t.dropna(subset=["ProvinceCode", "Tm"])
        .groupby("ProvinceCode", as_index=False)["Tm"]
        .mean()
        .rename(columns={"Tm": "StationAnnualMeanC"})
    )

    # ---- Fused province-level temperature ----
    merged_temp = station_agg.merge(weather_agg, on="ProvinceCode", how="outer")
    merged_temp["AverageAnnualTemperature"] = merged_temp[["StationAnnualMeanC", "WeatherAnnualMeanC"]].mean(axis=1)
    merged_temp["Province"] = merged_temp["ProvinceCode"].map(PROVINCE_CODE_TO_NAME)

    merged_temp = merged_temp.dropna(subset=["Province", "AverageAnnualTemperature"]).copy()
    merged_temp = merged_temp.sort_values("AverageAnnualTemperature").reset_index(drop=True)

    return merged_temp


def merge_datasets(cheese_df: pd.DataFrame, weather_df: pd.DataFrame) -> pd.DataFrame:
    """Merge cleaned cheese and weather data with explicit integrity checks."""
    print("Cheese dataframe shape BEFORE merge:", cheese_df.shape)
    print("Weather dataframe shape BEFORE merge:", weather_df.shape)

    merged = cheese_df.merge(
        weather_df[["ProvinceCode", "AverageAnnualTemperature"]],
        on="ProvinceCode",
        how="left",
        validate="many_to_one",
    )

    print("Merged dataframe shape AFTER merge:", merged.shape)
    print("Rows missing temperature after merge:", int(merged["AverageAnnualTemperature"].isna().sum()))

    assert len(merged) == len(cheese_df), "Data integrity issue: merge changed row count."
    return merged


def apply_premium_layout(fig: go.Figure, title: str, x_title: str, y_title: str) -> go.Figure:
    """Apply a consistent premium corporate visual style to Plotly figures."""
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=24, family=PREMIUM_FONT, color="#0f172a")),
        font=dict(family=PREMIUM_FONT, size=13, color="#1e293b"),
        paper_bgcolor="white",
        plot_bgcolor=PREMIUM_BG,
        margin=dict(t=95, r=35, b=80, l=75),
        width=1180,
        height=640,
        hoverlabel=dict(bgcolor="white", bordercolor="#cbd5e1", font_size=12),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
            title_text="",
        ),
    )
    fig.update_xaxes(title=x_title, showgrid=False, linecolor="#94a3b8", ticks="outside")
    fig.update_yaxes(title=y_title, showgrid=True, gridcolor=GRID_COLOR, zeroline=False, linecolor="#94a3b8", ticks="outside")
    return fig


# ------------------------
# Pipeline execution
# ------------------------
cheese_raw, weather_raw, temp_raw = load_data(".")
cheese_clean = clean_cheese_data(cheese_raw)
weather_clean = clean_weather_data(weather_raw, temp_raw)
analysis_df = merge_datasets(cheese_clean, weather_clean)

print("\n=== Clean cheese schema ===")
cheese_clean.info()
display(cheese_clean.head())

print("\n=== Clean weather schema ===")
weather_clean.info()
display(weather_clean.head())

print("\n=== Analytical dataset schema ===")
analysis_df.info()
display(analysis_df.head())

Cheese dataframe shape BEFORE merge: (1042, 18)
Weather dataframe shape BEFORE merge: (13, 5)
Merged dataframe shape AFTER merge: (1042, 19)
Rows missing temperature after merge: 0

=== Clean cheese schema ===
<class 'pandas.DataFrame'>
RangeIndex: 1042 entries, 0 to 1041
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   CheeseId              1042 non-null   int64  
 1   ManufacturerProvCode  1042 non-null   str    
 2   ManufacturingTypeEn   1042 non-null   str    
 3   MoisturePercent       1042 non-null   float64
 4   FlavourEn             801 non-null    str    
 5   CharacteristicsEn     643 non-null    str    
 6   Organic               1042 non-null   int64  
 7   CategoryTypeEn        1019 non-null   str    
 8   MilkTypeEn            1041 non-null   str    
 9   MilkTreatmentTypeEn   977 non-null    str    
 10  RindTypeEn            721 non-null    str    
 11  CheeseName            1042

,CheeseId,ManufacturerProvCode,ManufacturingTypeEn,MoisturePercent,FlavourEn,CharacteristicsEn,Organic,CategoryTypeEn,MilkTypeEn,MilkTreatmentTypeEn,RindTypeEn,CheeseName,FatLevel,ProvinceCode,Province,FatLevelClean,FatPercentEstimate,MilkTypeGroup
0,228,NB,Farmstead,47.0,"Sharp, lactic",Uncooked,0,Firm Cheese,Ewe,Raw Milk,Washed Rind,Sieur de Duplessis (Le),lower fat,NB,New Brunswick,lower fat,25.0,Sheep
1,242,NB,Farmstead,47.9,"Sharp, lactic, lightly caramelized",Uncooked,0,Semi-soft Cheese,Cow,Raw Milk,Washed Rind,Tomme Le Champ Doré,lower fat,NB,New Brunswick,lower fat,25.0,Cow
2,301,ON,Industrial,54.0,"Mild, tangy, and fruity","Pressed and cooked cheese, pasta filata, inter...",0,Firm Cheese,Cow,Pasteurized,NaN,Provolone Sette Fette (Tre-Stelle),lower fat,ON,Ontario,lower fat,25.0,Cow
3,303,NB,Farmstead,47.0,Sharp with fruity notes and a hint of wild honey,NaN,0,Veined Cheeses,Cow,Raw Milk,NaN,Geai Bleu (Le),lower fat,NB,New Brunswick,lower fat,25.0,Cow
4,319,NB,Farmstead,49.4,Softer taste,NaN,1,Semi-soft Cheese,Cow,Raw Milk,Washed Rind,Gamin (Le),lower fat,NB,New Brunswick,lower fat,25.0,Cow



=== Clean weather schema ===
<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ProvinceCode              13 non-null     str    
 1   StationAnnualMeanC        13 non-null     float64
 2   WeatherAnnualMeanC        13 non-null     float64
 3   AverageAnnualTemperature  13 non-null     float64
 4   Province                  13 non-null     str    
dtypes: float64(3), str(2)
memory usage: 652.0 bytes


,ProvinceCode,StationAnnualMeanC,WeatherAnnualMeanC,AverageAnnualTemperature,Province
0,NU,-11.812795,-11.625000,-11.718898,Nunavut
1,NT,-5.065504,-5.883333,-5.474419,Northwest Territories
2,YT,-2.614988,-2.183333,-2.399161,Yukon
3,MB,1.961404,-2.116667,-0.077631,Manitoba
4,SK,2.547995,1.933333,2.240664,Saskatchewan



=== Analytical dataset schema ===
<class 'pandas.DataFrame'>
RangeIndex: 1042 entries, 0 to 1041
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CheeseId                  1042 non-null   int64  
 1   ManufacturerProvCode      1042 non-null   str    
 2   ManufacturingTypeEn       1042 non-null   str    
 3   MoisturePercent           1042 non-null   float64
 4   FlavourEn                 801 non-null    str    
 5   CharacteristicsEn         643 non-null    str    
 6   Organic                   1042 non-null   int64  
 7   CategoryTypeEn            1019 non-null   str    
 8   MilkTypeEn                1041 non-null   str    
 9   MilkTreatmentTypeEn       977 non-null    str    
 10  RindTypeEn                721 non-null    str    
 11  CheeseName                1042 non-null   str    
 12  FatLevel                  1042 non-null   str    
 13  ProvinceCode              1042 non-null

,CheeseId,ManufacturerProvCode,ManufacturingTypeEn,MoisturePercent,FlavourEn,CharacteristicsEn,Organic,CategoryTypeEn,MilkTypeEn,MilkTreatmentTypeEn,RindTypeEn,CheeseName,FatLevel,ProvinceCode,Province,FatLevelClean,FatPercentEstimate,MilkTypeGroup,AverageAnnualTemperature
0,228,NB,Farmstead,47.0,"Sharp, lactic",Uncooked,0,Firm Cheese,Ewe,Raw Milk,Washed Rind,Sieur de Duplessis (Le),lower fat,NB,New Brunswick,lower fat,25.0,Sheep,4.882356
1,242,NB,Farmstead,47.9,"Sharp, lactic, lightly caramelized",Uncooked,0,Semi-soft Cheese,Cow,Raw Milk,Washed Rind,Tomme Le Champ Doré,lower fat,NB,New Brunswick,lower fat,25.0,Cow,4.882356
2,301,ON,Industrial,54.0,"Mild, tangy, and fruity","Pressed and cooked cheese, pasta filata, inter...",0,Firm Cheese,Cow,Pasteurized,NaN,Provolone Sette Fette (Tre-Stelle),lower fat,ON,Ontario,lower fat,25.0,Cow,6.445578
3,303,NB,Farmstead,47.0,Sharp with fruity notes and a hint of wild honey,NaN,0,Veined Cheeses,Cow,Raw Milk,NaN,Geai Bleu (Le),lower fat,NB,New Brunswick,lower fat,25.0,Cow,4.882356
4,319,NB,Farmstead,49.4,Softer taste,NaN,1,Semi-soft Cheese,Cow,Raw Milk,Washed Rind,Gamin (Le),lower fat,NB,New Brunswick,lower fat,25.0,Cow,4.882356


## Visualization 1: Climate vs Cheese Composition
### Bubble Scatter (Moisture vs Temperature, sized by Fat%)

- This chart examines whether warmer or colder provinces show concentration in moisture/fat profile ranges.
- Each bubble represents a cheese product, enriched with clean hover details for decision-ready interpretation.

In [3]:
viz1 = analysis_df.dropna(subset=["AverageAnnualTemperature", "MoisturePercent"]).copy()
viz1["FatPercentEstimate"] = viz1["FatPercentEstimate"].fillna(viz1["FatPercentEstimate"].median())

milk_order = ["Cow", "Goat", "Sheep", "Mixed", "Buffalo", "Other", "Unknown"]
milk_palette = {
    "Cow": "#1d4ed8",
    "Goat": "#f97316",
    "Sheep": "#16a34a",
    "Mixed": "#7c3aed",
    "Buffalo": "#92400e",
    "Other": "#475569",
    "Unknown": "#a3a3a3",
}

temp_med = viz1["AverageAnnualTemperature"].median()
moisture_med = viz1["MoisturePercent"].median()

fig1 = px.scatter(
    viz1,
    x="AverageAnnualTemperature",
    y="MoisturePercent",
    size="FatPercentEstimate",
    color="MilkTypeGroup",
    category_orders={"MilkTypeGroup": milk_order},
    color_discrete_map=milk_palette,
    size_max=25,
    opacity=0.80,
    custom_data=["CheeseName", "MilkTypeEn", "Province", "AverageAnnualTemperature", "MoisturePercent", "FatPercentEstimate"],
)

fig1.update_traces(
    marker=dict(line=dict(width=0.7, color="rgba(255,255,255,0.92)")),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Province: %{customdata[2]}<br>"
        "Milk Type: %{customdata[1]}<br>"
        "Avg Temperature: %{customdata[3]:.2f} C<br>"
        "Moisture: %{customdata[4]:.2f}%<br>"
        "Fat Estimate: %{customdata[5]:.1f}%<extra></extra>"
    ),
)

# Add a simple regression trendline for quick executive interpretation.
coef = np.polyfit(viz1["AverageAnnualTemperature"], viz1["MoisturePercent"], 1)
trend_x = np.linspace(viz1["AverageAnnualTemperature"].min(), viz1["AverageAnnualTemperature"].max(), 100)
trend_y = coef[0] * trend_x + coef[1]
fig1.add_trace(
    go.Scatter(
        x=trend_x,
        y=trend_y,
        mode="lines",
        name="Trendline",
        line=dict(color="#0f172a", width=2, dash="dot"),
        hovertemplate="Trendline<extra></extra>",
    )
)

fig1.add_vline(x=temp_med, line_dash="dash", line_color="#64748b", annotation_text="Median temperature", annotation_position="top left")
fig1.add_hline(y=moisture_med, line_dash="dash", line_color="#64748b", annotation_text="Median moisture", annotation_position="bottom right")

fig1 = apply_premium_layout(
    fig1,
    title="Moisture vs Temperature by Cheese (Bubble Size = Fat Estimate)",
    x_title="Average Annual Temperature (C)",
    y_title="Moisture Percentage (%)",
)

fig1.show()

## Visualization 2: Milk-Type Portfolio by Climate
### 100% Normalized Stacked Bar (Coldest to Warmest Provinces)

- Provinces are strictly ordered by cleaned average annual temperature.
- Bars are normalized to 100%, enabling direct comparison of milk-source composition regardless of production volume.

In [4]:
province_order = weather_clean[
    weather_clean["ProvinceCode"].isin(analysis_df["ProvinceCode"].unique())
].sort_values("AverageAnnualTemperature")["Province"].tolist()

portfolio = (
    analysis_df.groupby(["Province", "MilkTypeGroup"], as_index=False)
    .size()
    .rename(columns={"size": "CheeseCount"})
)

portfolio["ProvinceTotal"] = portfolio.groupby("Province")["CheeseCount"].transform("sum")
portfolio["SharePct"] = 100 * portfolio["CheeseCount"] / portfolio["ProvinceTotal"]
portfolio["LabelText"] = np.where(portfolio["SharePct"] >= 8, portfolio["SharePct"].round(1).astype(str) + "%", "")

milk_order = ["Cow", "Goat", "Sheep", "Mixed", "Buffalo", "Other", "Unknown"]
portfolio["Province"] = pd.Categorical(portfolio["Province"], categories=province_order, ordered=True)
portfolio["MilkTypeGroup"] = pd.Categorical(portfolio["MilkTypeGroup"], categories=milk_order, ordered=True)
portfolio = portfolio.sort_values(["Province", "MilkTypeGroup"])

milk_palette = {
    "Cow": "#1d4ed8",
    "Goat": "#f97316",
    "Sheep": "#16a34a",
    "Mixed": "#7c3aed",
    "Buffalo": "#92400e",
    "Other": "#475569",
    "Unknown": "#a3a3a3",
}

fig2 = px.bar(
    portfolio,
    x="Province",
    y="SharePct",
    color="MilkTypeGroup",
    color_discrete_map=milk_palette,
    category_orders={"MilkTypeGroup": milk_order, "Province": province_order},
    custom_data=["CheeseCount", "ProvinceTotal", "SharePct"],
    text="LabelText",
)

fig2.update_traces(
    textposition="inside",
    textfont=dict(color="white", size=11),
    cliponaxis=False,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Milk Type: %{fullData.name}<br>"
        "Cheese Count: %{customdata[0]} / %{customdata[1]}<br>"
        "Share: %{customdata[2]:.2f}%<extra></extra>"
    ),
)

fig2 = apply_premium_layout(
    fig2,
    title="Milk-Type Composition by Province (100% Normalized)",
    x_title="Province (Ordered: Coldest to Warmest)",
    y_title="Share of Province Cheese Portfolio (%)",
)
fig2.update_layout(
    barmode="stack",
    legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0),
)
fig2.update_xaxes(tickangle=-32)
fig2.update_yaxes(range=[0, 100], ticksuffix="%")

fig2.show()

## Visualization 3 (Bonus): Advanced Pattern Diagnostics
### Correlation Heatmap (Province-Level Feature Relationships)

- This chart moves from product-level records to **province-level aggregates**.
- It highlights structural relationships among climate, moisture, estimated fat intensity, and product volume.

In [5]:
province_metrics = (
    analysis_df.groupby(["Province", "ProvinceCode"], as_index=False)
    .agg(
        AvgTemperatureC=("AverageAnnualTemperature", "mean"),
        AvgMoisturePct=("MoisturePercent", "mean"),
        AvgFatPct=("FatPercentEstimate", "mean"),
        CheeseCount=("CheeseId", "count"),
        GoatShare=("MilkTypeGroup", lambda s: (s == "Goat").mean()),
        SheepShare=("MilkTypeGroup", lambda s: (s == "Sheep").mean()),
    )
)

corr_cols = [
    "AvgTemperatureC",
    "AvgMoisturePct",
    "AvgFatPct",
    "CheeseCount",
    "GoatShare",
    "SheepShare",
]
label_map = {
    "AvgTemperatureC": "Avg Temp",
    "AvgMoisturePct": "Avg Moisture",
    "AvgFatPct": "Avg Fat",
    "CheeseCount": "Cheese Count",
    "GoatShare": "Goat Share",
    "SheepShare": "Sheep Share",
}

corr_matrix = province_metrics[corr_cols].corr().rename(index=label_map, columns=label_map).round(3)

fig3 = go.Figure(
    data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.index,
        colorscale=[
            [0.0, "#67001f"],
            [0.2, "#b2182b"],
            [0.5, "#f7f7f7"],
            [0.8, "#2166ac"],
            [1.0, "#053061"],
        ],
        zmin=-1,
        zmax=1,
        zmid=0,
        text=corr_matrix.values,
        texttemplate="%{text:.2f}",
        textfont={"size": 12, "color": "#0f172a"},
        hovertemplate="%{y} vs %{x}: %{z:.3f}<extra></extra>",
        colorbar=dict(title="Correlation", tickvals=[-1, -0.5, 0, 0.5, 1]),
    )
)

fig3 = apply_premium_layout(
    fig3,
    title="Province-Level Correlation Matrix: Climate, Composition, and Output",
    x_title="Metrics",
    y_title="Metrics",
)
fig3.update_xaxes(side="bottom", tickangle=-20)
fig3.update_yaxes(autorange="reversed")

fig3.show()

display(province_metrics.sort_values("AvgTemperatureC").round(3))

,Province,ProvinceCode,AvgTemperatureC,AvgMoisturePct,AvgFatPct,CheeseCount,GoatShare,SheepShare
2,Manitoba,MB,-0.078,41.545,26.818,11,0.000,0.000
9,Saskatchewan,SK,2.241,17.000,45.000,1,0.000,0.000
4,Newfoundland and Labrador,NL,3.202,39.500,35.000,2,0.000,0.000
0,Alberta,AB,3.863,42.346,34.231,13,0.077,0.000
8,Quebec,QC,4.220,47.711,31.658,796,0.196,0.055
3,New Brunswick,NB,4.882,49.604,25.000,27,0.815,0.037
7,Prince Edward Island,PE,5.679,39.500,35.000,2,0.000,0.000
6,Ontario,ON,6.446,46.995,31.435,115,0.200,0.122
5,Nova Scotia,NS,6.613,41.300,35.000,10,0.000,0.200
1,British Columbia,BC,7.205,41.398,37.615,65,0.185,0.015


## Executive Conclusion

### Strategic Findings
- **Climate influence is present but not dominant:** Moisture and fat patterns show some directional variation across the temperature gradient, but relationships are generally moderate rather than deterministic.
- **Portfolio structure is shaped by regional systems:** Production concentration is uneven across provinces, and high-output regions exhibit broader product diversity that cannot be explained by climate alone.
- **Infrastructure and culture appear primary drivers:** The data supports a model where climate defines agricultural boundaries, while processing capability, local expertise, and historical cheese traditions govern product volume and mix.

### Senior-Level Interpretation
From a principal analytics perspective, provincial weather should be treated as an enabling context variable rather than a standalone causal engine. The visual evidence suggests that temperature may influence operating conditions and ingredient behavior at the margin, but regional industrial maturity and cultural identity are the stronger determinants of what gets produced at scale. In particular, provinces with established cheese-making ecosystems demonstrate that knowledge networks, facility investments, and market heritage can outweigh simple climatic constraints when determining portfolio depth.

### Recommended Next Steps
- Integrate farm-level and dairy supply-chain variables (feed, herd composition, processing capacity) to improve explanatory power.
- Add production volume and pricing data for weighted inference beyond product counts.
- Extend the model with multi-factor regression to isolate climate effects from structural regional factors.